# Structured outputs

Free-form text is the wrong contract for anything downstream — every consumer has to write a fragile parser. This notebook walks through four progressively stricter ways to get JSON out of an LLM:

1. **Free-form prose** — no contract at all.
2. **JSON-mode** — the model is told to emit JSON, but no schema is enforced.
3. **JSON-mode + Pydantic validation** — schema enforced *after* the call.
4. **Native structured outputs** (`response_format=json_schema`) — schema enforced by the provider during decoding.

All four use the shared `shared.llm` shim so the calls work against any provider via [litellm](https://docs.litellm.ai/) and are deterministically cached for CI.

In [1]:
# %% Notebook bootstrap — never touches API keys directly.
import os, sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / 'shared').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
# CI / offline mode: replay cached responses instead of calling out.
if not (os.getenv('OPENAI_API_KEY') or os.getenv('ANTHROPIC_API_KEY')):
    os.environ.setdefault('LLM_CACHE_ONLY', '1')
print('LLM_CACHE_ONLY =', os.environ.get('LLM_CACHE_ONLY', '0'))


LLM_CACHE_ONLY = 1


In [2]:
from shared.llm import Message, complete

MODEL = 'openai/gpt-4o-mini'
NS = '00-foundations/01-structured-outputs'

paper = (
    'Routing-Aware Sparse Mixture-of-Experts for Latency-Bounded Inference. '
    'Authors: A. Garcia, K. Sato, M. Owens (2024). The paper introduces RA-MoE, '
    'a routing scheme that biases experts based on activation history.'
)

## 1) Free-form prose
No schema. The output is human-readable but useless to a downstream consumer.

In [3]:
free = complete(
    model=MODEL,
    namespace=NS,
    messages=[
        Message(role='system', content='Summarize the paper in one short sentence.'),
        Message(role='user', content=paper),
    ],
)
print(free.content)

RA-MoE is a sparse mixture-of-experts routing scheme that biases experts by activation history to reduce latency.


## 2) JSON mode (no schema)
We instruct the model to emit JSON and set `response_format={'type':'json_object'}`. The output is JSON-shaped, but the model can still pick the wrong types or omit fields.

In [4]:
import json
extract_system = (
    'Extract the paper\'s title and the list of author names. '
    'Reply with ONLY a JSON object {"title": str, "authors": [str]}. '
    'No prose, no markdown.'
)
raw = complete(
    model=MODEL,
    namespace=NS,
    messages=[
        Message(role='system', content=extract_system),
        Message(role='user', content=paper),
    ],
    response_format={'type': 'json_object'},
).content
parsed = json.loads(raw)
parsed

{'title': 'Routing-Aware Sparse Mixture-of-Experts for Latency-Bounded Inference',
 'authors': ['A. Garcia', 'K. Sato', 'M. Owens']}

## 3) JSON mode + Pydantic post-validation
Pydantic catches type errors and missing fields — and produces a usable Python object instead of a `dict[str, Any]`. The cost is one extra call site, not an extra LLM call.

In [5]:
from pydantic import BaseModel, ValidationError

class PaperMeta(BaseModel):
    title: str
    authors: list[str]

meta = PaperMeta.model_validate_json(raw)
print(meta)
print('title:', meta.title)
print('first author:', meta.authors[0])

title='Routing-Aware Sparse Mixture-of-Experts for Latency-Bounded Inference' authors=['A. Garcia', 'K. Sato', 'M. Owens']
title: Routing-Aware Sparse Mixture-of-Experts for Latency-Bounded Inference
first author: A. Garcia


In [6]:
# A deliberately broken response shows Pydantic catching the type error.
broken = complete(
    model=MODEL,
    namespace=NS,
    messages=[
        Message(role='system', content=extract_system),
        Message(role='user', content='Some malformed example.'),
    ],
    response_format={'type': 'json_object'},
).content
try:
    PaperMeta.model_validate_json(broken)
except ValidationError as e:
    print('Pydantic caught it:')
    print(e)

Pydantic caught it:
1 validation error for PaperMeta
authors
  Input should be a valid array [type=list_type, input_value='not a list', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/list_type


## 4) Native structured outputs (`response_format=json_schema`)
When the provider supports it (OpenAI, recent Anthropic, Mistral), passing a JSON-schema in `response_format` forces the *decoder* to emit valid JSON for that schema. The reply is guaranteed to parse — no need for a try/except around `json.loads`.

We still validate with Pydantic so the rest of our code sees a typed object.

In [7]:
class PaperMetaFull(BaseModel):
    title: str
    authors: list[str]
    year: int

schema = {
    'type': 'object',
    'additionalProperties': False,
    'required': ['title', 'authors', 'year'],
    'properties': {
        'title': {'type': 'string'},
        'authors': {'type': 'array', 'items': {'type': 'string'}},
        'year': {'type': 'integer'},
    },
}
native = complete(
    model=MODEL,
    namespace=NS,
    messages=[
        Message(role='system', content='Extract metadata from the paper.'),
        Message(role='user', content=paper),
    ],
    response_format={
        'type': 'json_schema',
        'json_schema': {'name': 'PaperMetadata', 'schema': schema, 'strict': True},
    },
).content
meta_full = PaperMetaFull.model_validate_json(native)
meta_full

PaperMetaFull(title='Routing-Aware Sparse Mixture-of-Experts for Latency-Bounded Inference', authors=['A. Garcia', 'K. Sato', 'M. Owens'], year=2024)

## Comparison

| Pattern | Guarantees JSON? | Guarantees schema? | Provider support |
|---|---|---|---|
| Free-form | ❌ | ❌ | universal |
| JSON-mode | ✅ | ❌ (model picks types) | OpenAI, Anthropic, most |
| JSON-mode + Pydantic | ✅ | ✅ (post-hoc) | universal |
| Native (`json_schema`) | ✅ | ✅ (at decode time) | OpenAI, recent Anthropic, Mistral |

**Default to native structured outputs when the provider supports it**, fall back to JSON-mode + Pydantic everywhere else. Never trust free-form prose for anything downstream.